In [1]:
# -*- coding: utf-8 -*-
# Left-join metadata onto main without bringing duplicate column names or excluded metadata fields.

import os
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse

BASE_DIR   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
MAIN_PATH  = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
META_PATH  = os.path.join(BASE_DIR, "Project_Metadata.csv")
OUT_PATH   = MAIN_PATH  # overwrite

FETCH_DATE = datetime(2025, 8, 10)

# Columns from metadata to explicitly exclude
EXCLUDE_META_COLS = {
    "html_url", "repo_index", "repo_name", "id", "name", "full_name", "owner"
}

def norm_key_main(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
              .str.replace("/", ".", regex=False)
              .str.strip()
              .str.lower()
    )

def owner_repo_from_html_url(url: str) -> tuple[str, str]:
    try:
        path = urlparse(str(url)).path.strip("/")
        parts = path.split("/")
        if len(parts) >= 2:
            return parts[0].strip(), parts[1].strip()
    except Exception:
        pass
    return "", ""

def meta_key_from_html(series: pd.Series) -> pd.Series:
    return series.astype(str).map(lambda u: ".".join(owner_repo_from_html_url(u))).str.lower()

# --- Load ---
main = pd.read_csv(MAIN_PATH, dtype=str).fillna("")
meta = pd.read_csv(META_PATH, dtype=str).fillna("")

# --- Validate ---
if "full_name" not in main.columns:
    raise ValueError("Main file must contain a 'full_name' column.")
if "html_url" not in meta.columns:
    raise ValueError("Metadata file must contain an 'html_url' column.")

# --- Keys ---
main["__key__"] = norm_key_main(main["full_name"])
meta["__key__"] = meta_key_from_html(meta["html_url"])

# --- repo_age ---
if "created_at" in meta.columns:
    created_dt = pd.to_datetime(meta["created_at"], errors="coerce", utc=True)
    meta["repo_age"] = ((pd.to_datetime(FETCH_DATE) - created_dt.dt.tz_localize(None)).dt.days / 365.25).round(2)
else:
    meta["repo_age"] = ""

# --- Filter metadata columns ---
main_cols_set = set(main.columns)
meta_cols_to_add = [
    c for c in meta.columns
    if c not in main_cols_set and c != "__key__" and c not in EXCLUDE_META_COLS
]

meta_to_merge = meta[["__key__"] + meta_cols_to_add]

# --- Merge ---
out = main.merge(meta_to_merge, on="__key__", how="left").drop(columns=["__key__"], errors="ignore")

# --- Save ---
out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH} (rows={len(out)}, metadata_cols_added={len(meta_cols_to_add)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518, metadata_cols_added=0)
